## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and downloads the data — just press ▶ and wait for
the green **✅ Setup complete**, then run the rest of the notebook top to bottom.

It also offers to connect your Google Drive so your figures are *saved* for your
poster (recommended). If you skip that, the notebook still works — your figures
just won't persist after you close Colab.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  downloading the data (~470 MB, first time only) ...")
import gdown
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/synapse_preprocessed.pkl"):
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY",
                   output="data/synapse_preprocessed.pkl", quiet=False)
os.environ["CAMP_DATA_PATH"] = "data/synapse_preprocessed.pkl"

# Save figures to your own Drive so they persist for your poster (recommended).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    where = "Drive > DecodingBrain_outputs"
except Exception:
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    where = "a temporary 'outputs' folder (download anything you want to keep!)"
print(f"\n\u2705 Setup complete. Figures will be saved to {where}.")


# Week 1 · Day 4 — Brain Rhythms (Spectral Power)

The ERP showed the brain's response *over time*. Now we look at the same signal
a completely different way: as a mix of **rhythms** at different speeds. This is
the **frequency domain**, and it's measured with **Power Spectral Density (PSD)**.

Brain rhythms are grouped into named **bands**:

| Band | Speed (Hz) | Loosely associated with |
|---|---|---|
| delta | 1–4 | deep states, large slow waves |
| theta | 4–8 | drowsiness, memory |
| alpha | 8–13 | relaxed, "idling" |
| beta | 13–30 | active thinking, alertness |
| gamma | 30–50 | intense processing |

### By the end of this notebook you will be able to
1. Compute a power spectrum and read it
2. Understand **baseline normalization** (the before/after trick)
3. Build a band-power calculator yourself
4. See which rhythms change when a sound plays

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
print("Frequency bands:", cu.BANDS)

## 1. What does a spectrum look like?
MNE can compute the PSD of any epochs with `.compute_psd()`. Let's see the full
spectrum during the sound for one subject.

In [ ]:
epochs = data["exp_epochs"]["let"][0]
subject = data["exp_subjects"][0]

# only keep the 2 seconds while the sound plays (0 to 2 s)
during_sound = epochs.copy().crop(tmin=0, tmax=2)
psd = during_sound.compute_psd(method="welch", fmin=1, fmax=50, verbose=False)

freqs = psd.freqs
power = psd.get_data().mean(axis=(0, 1))   # average over epochs and channels

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(freqs, power, color="black")
# shade the bands
for band, (lo, hi) in cu.BANDS.items():
    ax.axvspan(lo, hi, alpha=0.15, color=cu.BAND_COLORS[band], label=band)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power")
ax.set_title(f"{subject} — power spectrum during sound")
ax.legend(ncol=5, fontsize=8)
plt.show()

Notice the curve is highest at low frequencies and drops off — that's normal for
brain signals. The colored bands show where each rhythm lives.

## 2. The key idea: baseline normalization
Raw power is hard to compare across people (everyone's electrodes sit a little
differently). The fix: for each person, compare power **during** the sound to
power in their own **quiet baseline** before the sound. We report the change in
**decibels (dB)**:

$$ \text{dB} = 10 \times \log_{10}\!\left(\frac{\text{power during sound}}{\text{power in baseline}}\right) $$

- dB **> 0** → that rhythm got **stronger** when the sound played
- dB **< 0** → that rhythm got **weaker**
- dB **= 0** → no change

This "talk in terms of change from baseline" is exactly what the real study does.

## 3. Build the band-power calculator yourself
This is the most important function of the week. Fill in the three blanks. The
structure is given; you supply the cropping and the dB formula.

In [ ]:
def my_band_power_db(epochs, band, stim_window, baseline_window=cu.BASELINE_WINDOW):
    """Return baseline-normalized power for one band, in dB."""
    fmin, fmax = cu.BANDS[band]
    n_fft = min(64, int(epochs.info["sfreq"] * 0.5))

    # (a) TODO: crop out the baseline window and the stim window
    #     hint: epochs.copy().crop(tmin=..., tmax=...)
    baseline = None   # epochs.copy().crop(tmin=baseline_window[0], tmax=baseline_window[1])
    stim = None       # epochs.copy().crop(tmin=stim_window[0], tmax=stim_window[1])

    # compute power in each (this part is done for you)
    bl_power = baseline.compute_psd(method="welch", fmin=fmin, fmax=fmax,
                                    n_fft=n_fft, verbose=False).get_data().mean()
    st_power = stim.compute_psd(method="welch", fmin=fmin, fmax=fmax,
                                n_fft=n_fft, verbose=False).get_data().mean()

    # (b) TODO: return the dB change using the formula above
    #     hint: 10 * np.log10(st_power / bl_power)
    return None


# --- test it against the official version ---
win = cu.get_time_windows("let")["full_stim"]
mine = my_band_power_db(epochs, "gamma", win)
official = cu.band_power_db(epochs, "gamma", win)
cu.check(mine is not None and abs(mine - official) < 1e-6,
         f"Your function gives {mine:.3f} dB — matches the official version!" if mine else "",
         "Crop baseline & stim, then return 10*np.log10(st_power/bl_power).")

From here on we'll use `cu.band_power_db()` (same math, fully tested) so
everyone's numbers line up.

## 4. All five bands for one subject
Let's compute the dB change in every band for our subject during the sound.

In [ ]:
win = cu.get_time_windows("let")["full_stim"]   # the full 0–2 s sound window
for band in cu.BAND_ORDER:
    db = cu.band_power_db(epochs, band, win)
    arrow = "↑ stronger" if db > 0 else "↓ weaker"
    print(f"  {band:6s}: {db:+.2f} dB   {arrow}")

### ✏️ Your turn #1
Compute the **alpha** dB change for the **first CTRL subject** in the **`hlt`**
task. Use the HLT `full_stim` window.

*Hint:* get the window with `cu.get_time_windows("hlt")["full_stim"]`.

In [ ]:
ctrl_ep = data["ctrl_epochs"]["hlt"][0]
# TODO: compute alpha dB change
hlt_win = None
alpha_db = None

cu.check(alpha_db is not None,
         f"Alpha change = {alpha_db:.2f} dB" if alpha_db is not None else "",
         "Get the hlt full_stim window, then call cu.band_power_db(ctrl_ep, 'alpha', hlt_win).")

## 5. Compare the groups, band by band
Now the science: for the LET task, compute each subject's gamma change, then
compare the EXP and CTRL groups. (Gamma is interesting because the "central gain"
theory of hyperacusis predicts the sound-sensitive brain over-amplifies signals.)

In [ ]:
exp_gamma = [cu.band_power_db(ep, "gamma", win)
             for _, ep in cu.iter_subjects(data, "exp", "let")]
ctrl_gamma = [cu.band_power_db(ep, "gamma", win)
              for _, ep in cu.iter_subjects(data, "ctrl", "let")]

exp_gamma = [x for x in exp_gamma if not np.isnan(x)]
ctrl_gamma = [x for x in ctrl_gamma if not np.isnan(x)]

print(f"EXP  gamma change: mean = {np.mean(exp_gamma):+.2f} dB  (n={len(exp_gamma)})")
print(f"CTRL gamma change: mean = {np.mean(ctrl_gamma):+.2f} dB  (n={len(ctrl_gamma)})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.boxplot([exp_gamma, ctrl_gamma], labels=["EXP", "CTRL"])
# scatter the individual subjects on top
for i, vals in enumerate([exp_gamma, ctrl_gamma], start=1):
    jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
    color = cu.EXP_COLOR if i == 1 else cu.CTRL_COLOR
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=color, alpha=0.7, zorder=3)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Gamma change from baseline (dB)")
ax.set_title("LET gamma power: EXP vs CTRL")
plt.show()

### ✏️ Your turn #2
Pick a **different band** (try `"alpha"` or `"beta"`) and make the same boxplot
for the LET task. Does that band separate the groups more or less than gamma?

In [ ]:
my_band = "alpha"   # TODO: try a few
# TODO: compute exp_vals and ctrl_vals for my_band (copy the pattern above),
#       drop NaNs, then make the boxplot.

## 🎯 Wrap-up
You can now turn a brain recording into rhythm strengths, normalize to baseline,
and compare groups band by band. This `band_power_db` is a **feature** — a single
number summarizing a recording. Next week you'll compute hundreds of them.

➡️ **Next:** Notebook 04 — your first full EXP-vs-CTRL sweep across all tasks
and bands (**Checkpoint 1**).